In [ ]:
# Install playwright, its system dependencies, and the chromium browser
!pip install playwright
!playwright install-deps chromium
!playwright install chromium

In [6]:
locations = [
    "others",
    "alexandria",
    "6th-settlement",
    "northern-expansion",
    "el-gouna",
    "giza",
    "north-coast-sahel",
    "el-shorouk",
    "el-choueifat",
    "new-zayed",
    "el-sheikh-zayed",
    "al-dabaa",
    "new-capital-city",
    "al-alamein",
    "ain-sokhna",
    "hurghada",
    "new-cairo",
    "old-cairo",
    "central-cairo",
    "el-lotus",
    "south-investors",
    "north-investors",
    "maadi",
    "port-said",
    "south-new-cairo",
    "golden-square",
    "october-gardens",
    "new-capital-gardens",
    "ras-el-hekma",
    "ras-sudr",
    "new-sphinx",
    "sahl-hasheesh",
    "somabay",
    "sidi-heneish",
    "sidi-abdel-rahman",
    "alam-al-roum",
    "ghazala-bay",
    "6th-of-october-city",
    "mostakbal-city",
    "madinaty",
    "mokattam",
    "makadi",
    "new-mansoura",
    "new-heliopolis",
    "heliopolis",
    "downtown-cairo"
]

In [5]:
import asyncio
from bs4 import BeautifulSoup
from playwright.async_api import async_playwright


# Define the target URL for the Nawy 6th Settlement area page
url = "https://www.nawy.com/area/6th-settlement"

async def extract_area_text():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)

        # Open a new browser page
        page = await browser.new_page()

        # Navigate to the target URL
        await page.goto(url)

        # Wait until the 'head' container (which holds the description) is loaded in the DOM
        await page.wait_for_selector("#head")

        # The description text is often truncated with a "See More" button.
        # We'll look for the button and click it to expand the full text before extracting.
        try:
            # Wait briefly for the button. The data-test attribute was found in the HTML snippet.
            show_more_btn = page.locator('button[data-test="show-more-less"]')

            # If the button exists and is visible, interact with it
            if await show_more_btn.is_visible(timeout=3000):
                btn_text = await show_more_btn.text_content()

                # Check if it actually says "See More" (to avoid clicking "See Less")
                if btn_text and "More" in btn_text:
                    print("Clicking 'See More' to expand the text...")
                    await show_more_btn.click()

                    # Wait a moment for the DOM to update with the full text
                    await page.wait_for_timeout(1500)
        except Exception as e:
            print("No 'See More' button found or needed.")

        # ================================
        # EXTRACT FULL PAGE HTML
        # ================================
        html_content = await page.content()

        # Close the browser once we have the HTML
        await browser.close()

    return html_content

# In Jupyter, we can directly await the function to run it.
# (Note: If running as a standard python script outside Jupyter, use `html_content = asyncio.run(extract_area_text())` instead)
html_content = await extract_area_text()

# Parse the HTML using BeautifulSoup for easier data extraction
soup = BeautifulSoup(html_content, 'html.parser')

# Find the specific container holding the area text (from the provided HTML snippet)
head_container = soup.find('div', id='head')

if head_container:
    # Extract all text, separating elements with a newline and stripping extra whitespace
    extracted_text = head_container.get_text(separator='\n', strip=True)

    print("--- EXTRACTED TEXT ---")
    print(extracted_text)

    # Save the extracted text to a file
#     with open('6th_settlement_description.txt', 'w', encoding='utf-8') as f:
#         f.write(extracted_text)
#     print("\nText successfully saved to 6th_settlement_description.txt")
# else:
#     print("Could not find the target container with id='head'.")

--- EXTRACTED TEXT ---
About 6th settlement
6th Settlement is one of the elite neighborhoods in New Cairo City. It has been formerly called “Al Aml Triangle” and its name has been changed since it is located at the endpoint of 5th Settlement.
The New Urban Communities Authority is in charge of developing this area under the supervision of the Ministry of Housing, Utilities, and Urban Communities, and it works closely with New Cairo’s Development Authority.
New Cairo’s 6th Settlement is one of the most recent upscale residential areas, offering a premium location and many opulent compounds. The area stands out as a residential, administrative, and commercial extension of New Cairo City.
6th Settlement New Cairo Egypt, extends over 4800 acres (20 million square meters) of land to the east of Cairo. Also, it is privileged with the prime location that makes it easily accessible from all the major roads.
6th Settlement Location In New Cairo
6th Settlement enjoys a prime location in New Cair

In [7]:
async def extract_all_areas():
    # Use Playwright to launch a Chromium browser instance for full JavaScript rendering
    async with async_playwright() as p:
        # Launch the browser in headless mode (Required for Google Colab/Linux servers)
        browser = await p.chromium.launch(headless=True)
        # Open a single browser page to reuse for all locations
        page = await browser.new_page()

        # Initialize/clear the master text file before starting
        with open('all_locations_descriptions.txt', 'w', encoding='utf-8') as f:
            f.write("NAWY LOCATIONS DESCRIPTIONS\n")

        for location in locations:
            url = f"https://www.nawy.com/area/{location}"
            print(f"\n--- Processing: {location} ---")

            try:
                # Navigate to the target URL
                await page.goto(url)

                # Wait until the 'head' container is loaded, with a timeout of 10 seconds to avoid hanging on bad pages
                await page.wait_for_selector("#head", timeout=10000)

                # Look for the 'See More' button to expand truncated text
                try:
                    show_more_btn = page.locator('button[data-test="show-more-less"]')

                    if await show_more_btn.is_visible(timeout=3000):
                        btn_text = await show_more_btn.text_content()
                        if btn_text and "More" in btn_text:
                            print(f"[{location}] Clicking 'See More' to expand the text...")
                            await show_more_btn.click()
                            await page.wait_for_timeout(1500)
                except Exception:
                    # Ignore if no button is found, it just means the text isn't truncated
                    pass

                # Extract the full HTML content for this page
                html_content = await page.content()

                # Parse the HTML using BeautifulSoup
                soup = BeautifulSoup(html_content, 'html.parser')
                head_container = soup.find('div', id='head')

                if head_container:
                    # Extract text and clean it
                    extracted_text = head_container.get_text(separator='\n', strip=True)

                    # Append the extracted text to the single master file
                    with open('all_locations_descriptions.txt', 'a', encoding='utf-8') as f:
                        f.write(f"\n\n{'='*50}\nLOCATION: {location.upper()}\n{'='*50}\n\n")
                        f.write(extracted_text)

                    print(f"[{location}] Successfully appended to all_locations_descriptions.txt")
                else:
                    print(f"[{location}] Could not find the target container with id='head'.")

            except Exception as e:
                print(f"[{location}] Error extracting data: {e}")

        # Close the browser once all locations are processed
        await browser.close()
        print("\nAll locations processed!")

# In Jupyter, we can directly await the function to run it.
# (Note: If running as a standard python script outside Jupyter, use `asyncio.run(extract_all_areas())` instead)
await extract_all_areas()


--- Processing: others ---
[others] Error extracting data: Page.wait_for_selector: Timeout 10000ms exceeded.
Call log:
  - waiting for locator("#head") to be visible


--- Processing: alexandria ---
[alexandria] Successfully appended to all_locations_descriptions.txt

--- Processing: 6th-settlement ---
[6th-settlement] Successfully appended to all_locations_descriptions.txt

--- Processing: northern-expansion ---
[northern-expansion] Successfully appended to all_locations_descriptions.txt

--- Processing: el-gouna ---
[el-gouna] Successfully appended to all_locations_descriptions.txt

--- Processing: giza ---
[giza] Error extracting data: Page.wait_for_selector: Timeout 10000ms exceeded.
Call log:
  - waiting for locator("#head") to be visible


--- Processing: north-coast-sahel ---
[north-coast-sahel] Successfully appended to all_locations_descriptions.txt

--- Processing: el-shorouk ---
[el-shorouk] Successfully appended to all_locations_descriptions.txt

--- Processing: el-choueifa